# Entrenamiento por lotes

## Pasos
1. Cargar datos
2. Obtener batches de 1000 muestras normalmente distribuidas (etiquetas misma distribución)
3. Entrenar por lotes
4. Promediar los gradientes y actualizar los pesos

## Caso 1
Entrenar por lotes entrenando cada uno hasta alcanzar un minimo de 80% de precisión en el conjunto de entrenamiento.
Continuar con el otro batch y promediar los gradientes y actualizar los pesos al final del entrenamiento.

## Caso 2
Entrenar por lotes entrenando cada uno solo 1 iteración por N épocas, promediar los gradientes y actualizar los pesos antes de continuar la siguiente época.

# Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

RUN_ID = 2
BASE_PATH = "/content/drive/MyDrive/Colab Notebooks/MNIST"

def get_save_path(name):
    path = os.path.join(BASE_PATH, name, f"run_{RUN_ID}")
    os.makedirs(path, exist_ok=True)

    return path

# Librerías usadas

In [4]:
from kagglehub import dataset_download
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.metrics import confusion_matrix

# CARGA Y PREPARACIÓN DE DATOS

In [5]:
print("Descargando dataset...")
path = dataset_download("oddrationale/mnist-in-csv")

train_df = pd.read_csv(path + "/mnist_train.csv")
test_df = pd.read_csv(path + "/mnist_test.csv")

Descargando dataset...
Using Colab cache for faster access to the 'mnist-in-csv' dataset.


In [6]:
train_df.head()

# Separar características (X) y etiquetas (y)
X_train = train_df.iloc[:, 1:].values  # 60000 imágenes de 784 píxeles (28x28)
y_train = train_df.iloc[:, 0].values   # 60000 etiquetas (0-9)
X_test = test_df.iloc[:, 1:].values
y_test = test_df.iloc[:, 0].values

# Normalizar: los píxeles van de 0-255, los llevamos a 0-1
# Esto ayuda a que los gradientes no exploten y el entrenamiento sea estable
X_train = X_train / 255.0
X_test = X_test / 255.0

print(f"X_train shape: {X_train.shape} {X_train.dtype}")    # (60000, 784)
print(f"y_train shape: {y_train.shape} {y_train.dtype}")    # (60000,)
print(f"X_test shape: {X_test.shape} {X_test.dtype}")      # (10000, 784)
print(f"y_test shape: {y_test.shape} {y_test.dtype}")      # (10000,)

X_train shape: (60000, 784) float64
y_train shape: (60000,) int64
X_test shape: (10000, 784) float64
y_test shape: (10000,) int64


# ARQUITECTURA DE LA RED NEURONAL

In [13]:
INPUT_SIZE = 784
HIDDEN_SIZE = 10
OUTPUT_SIZE = 10
LEARNING_RATE = 0.05
EPOCHS = 200
BATCH_SIZE = 6000
SEED = 42 + RUN_ID
N_BATCHES = X_train.shape[0] // BATCH_SIZE

# INICIALIZACIÓN DE PESOS Y SESGOS

In [14]:
np.random.seed(seed=SEED)

w1_init = np.random.randn(INPUT_SIZE, HIDDEN_SIZE) * 0.01
b1_init = np.zeros((1, HIDDEN_SIZE))

w2_init = np.random.randn(HIDDEN_SIZE, OUTPUT_SIZE) * 0.01
b2_init = np.zeros((1, OUTPUT_SIZE))

np.random.seed()

# FUNCIONES

In [9]:
def relu(z):
    """
    Rectified Linear Unit: max(0, z)
    Si z > 0, devuelve z. Si z <= 0, devuelve 0.
    Es no lineal, simple y evita el problema del gradiente desvaneciente.
    """
    return np.maximum(0, z)

def relu_derivative(z):
    """
    Derivada de ReLU: 1 si z > 0, 0 si z <= 0
    Esto se usa en backpropagation para calcular gradientes.
    """
    return (z > 0).astype(float)

def softmax(z):
    """
    Softmax: convierte logits en probabilidades que suman 1.
    z puede ser cualquier número, softmax lo normaliza.
    Se resta el max para estabilidad numérica (evita overflow).
    """
    exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)

def mse_loss(y_pred, y_true_one_hot):
    """
    Error Cuadrático Medio: MSE = (1/n) * Σ(y_pred - y_true)²
    Mide qué tan lejos están las predicciones de los valores reales.
    """
    return np.mean((y_pred - y_true_one_hot) ** 2)

def mse_derivative(y_pred, y_true_one_hot):
    """
    Derivada del MSE respecto a y_pred: 2*(y_pred - y_true)/n
    Esto nos dice en qué dirección ajustar para reducir el error.
    """
    n = y_pred.shape[0]
    return 2 * (y_pred - y_true_one_hot) / n

def one_hot(y, num_classes=10):
    """
    Convierte etiquetas [0, 5, 3, ...] en vectores one-hot:
    0 -> [1, 0, 0, 0, 0, 0, 0, 0, 0, 0]
    5 -> [0, 0, 0, 0, 0, 1, 0, 0, 0, 0]
    Necesario para comparar con la salida de softmax (10 probabilidades).
    """
    one_hot_matrix = np.zeros((len(y), num_classes))
    one_hot_matrix[np.arange(len(y)), y] = 1
    return one_hot_matrix


# Forward y Backward Pass

In [10]:
def forward(X):
    """
    Calcula la salida de la red paso a paso.
    Guardamos valores intermedios (z1, a1, etc.) para usarlos en backprop.

    X: matriz de entrada (batch_size, 784)
    Entrada -> Capa oculta (ReLU) -> Salida (Softmax)

    Retorna: salida final y valores intermedios para backprop
    """
    # Capa 1: Entrada -> Oculta 1
    z1 = np.dot(X, w1) + b1      # Suma ponderada: X·W + b # que piensa la capa sin activación
    a1 = relu(z1)                # Activación ReLU         # que piensa la capa con activación

    # Capa 2: Oculta 1 -> Salida
    z2 = np.dot(a1, w2) + b2
    a2 = softmax(z2)             # Probabilidades para cada clase # que piensa toda la red neuronal

    # Guardamos todo en un diccionario para el backprop
    cache = {
        'z1': z1, 'a1': a1,
        'z2': z2, 'a2': a2,
        'X': X
    }

    return a2, cache

def backward(cache, y_true_one_hot, w1, w2, b1, b2):
    X = cache['X']
    z1 = cache['z1']
    a1 = cache['a1']
    z2 = cache['z2']
    output = cache['a2']

    m = len(X)  # tamaño del batch

    # Error capa salida
    dz2 = output - y_true_one_hot

    # Gradientes capa salida
    dw2 = np.dot(a1.T, dz2) / m
    db2 = np.sum(dz2, axis=0, keepdims=True) / m

    # Backprop a capa oculta
    dz1 = np.dot(dz2, w2.T) * relu_derivative(z1)

    dw1 = np.dot(X.T, dz1) / m
    db1 = np.sum(dz1, axis=0, keepdims=True) / m

    return dw1, dw2, db1, db2


# ENTRENAMIENTO (GRADIENT DESCENT NO ESTOCÁSTICO)

In [11]:
def train_batch(X_batch, y_batch, w1, w2, b1, b2):

    # Forward
    output, cache = forward(X_batch)

    y_one_hot = one_hot(y_batch)
    loss = mse_loss(output, y_one_hot)

    # Backward
    dw1, dw2, db1, db2 = backward(cache, y_one_hot, w1, w2, b1, b2)

    # NORMA DEL GRADIENTE
    grad_norm = np.sqrt(
        np.sum(dw1**2) +
        np.sum(dw2**2) +
        np.sum(db1**2) +
        np.sum(db2**2)
    )

    w1 -= LEARNING_RATE * dw1
    w2 -= LEARNING_RATE * dw2
    b1 -= LEARNING_RATE * db1
    b2 -= LEARNING_RATE * db2

    predictions = np.argmax(output, axis=1)
    accuracy = np.mean(predictions == y_batch)

    return loss, accuracy, grad_norm


def predict(X):
    """Predice clases para nuevos datos."""
    output, _ = forward(X)
    return np.argmax(output, axis=1)

# Método 1

In [15]:
N_SUBSETS = N_BATCHES          # Número de nodos / particiones
LOCAL_TARGET_ACC = 0.8         # Criterio local
MAX_LOCAL_UPDATES = 500        # Máximo por subconjunto

print("\n" + "="*50)
print("INICIANDO ENTRENAMIENTO - MODEL AVERAGING")
print(f"Run: {RUN_ID}")
print(f"Seed: {SEED}")
print("="*50)
print(f"Subconjuntos (nodos): {N_SUBSETS}")
print(f"Muestras totales: {len(X_train)}")
print("="*50)

SAVE_PATH = get_save_path("Metodo_1")


# Mezclar una sola vez
indices = np.random.permutation(len(X_train))
X_shuffled = X_train[indices]
y_shuffled = y_train[indices]

# Dividir en subconjuntos disjuntos
subset_size = len(X_train) // N_SUBSETS

subsets = []
for k in range(N_SUBSETS):
    start = k * subset_size
    end = (k + 1) * subset_size if k < N_SUBSETS - 1 else len(X_train)
    subsets.append((X_shuffled[start:end], y_shuffled[start:end]))

# entrenamiento
local_models = []
local_metrics = []

for k, (X_local, y_local) in enumerate(subsets):
    print(f"\n--- Entrenando Subconjunto {k+1}/{N_SUBSETS} ---")

    # Inicialización independiente
    w1, b1 = w1_init.copy(), b1_init.copy()
    w2, b2 = w2_init.copy(), b2_init.copy()

    local_updates = 0
    local_loss_sum = 0
    local_acc_sum = 0
    local_grad_sum = 0

    local_acc = 0

    while local_acc < LOCAL_TARGET_ACC and local_updates < MAX_LOCAL_UPDATES:
        loss, acc, grad_norm = train_batch(
            X_local, y_local, w1, w2, b1, b2
        )

        local_acc = acc
        local_updates += 1

        local_loss_sum += loss
        local_acc_sum += acc
        local_grad_sum += grad_norm

        if local_updates % 50 == 0 or local_updates == 1:
            print(f"Subset {k+1} | Update {local_updates} | "
                  f"Loss {loss:.4f} | Acc {acc:.4f}")

    # Guardar datos modelo entrenado
    local_models.append((w1.copy(), b1.copy(), w2.copy(), b2.copy()))

    local_metrics.append({
        "subset": k+1,
        "updates": local_updates,
        "avg_loss": local_loss_sum / local_updates,
        "avg_acc": local_acc_sum / local_updates,
        "avg_grad_norm": local_grad_sum / local_updates
    })

# Promediar pesos
# Promedio de parámetros
w1_avg = np.mean([m[0] for m in local_models], axis=0)
b1_avg = np.mean([m[1] for m in local_models], axis=0)
w2_avg = np.mean([m[2] for m in local_models], axis=0)
b2_avg = np.mean([m[3] for m in local_models], axis=0)

# Reemplazar modelo global
w1, b1 = w1_avg, b1_avg
w2, b2 = w2_avg, b2_avg

# evaluación
final_train_preds = predict(X_train)
final_test_preds = predict(X_test)

train_acc = np.mean(final_train_preds == y_train)
test_acc = np.mean(final_test_preds == y_test)

print("\n" + "="*50)
print("RESULTADO FINAL MODEL AVERAGING")
print(f"Train Accuracy: {train_acc:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print("="*50)

# metricas
local_df = pd.DataFrame(local_metrics)
local_df.to_csv(os.path.join(SAVE_PATH, "local_training_metrics.csv"), index=False)

evaluation_df = pd.DataFrame({
    "train_accuracy_final": [train_acc],
    "test_accuracy_final": [test_acc],
    "n_subsets": [N_SUBSETS]
})

evaluation_df.to_csv(os.path.join(SAVE_PATH, "evaluation.csv"), index=False)

plt.figure()
plt.plot(local_df["subset"], local_df["avg_acc"])
plt.xlabel("Subconjunto")
plt.ylabel("Accuracy promedio local")
plt.title("Rendimiento por nodo")
plt.grid(True)
plt.savefig(os.path.join(SAVE_PATH, "local_performance.png"))
plt.close()

from sklearn.metrics import confusion_matrix
import seaborn as sns

cm = confusion_matrix(y_test, final_test_preds)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.title("Matriz de Confusión - Modelo Promediado")
plt.xlabel("Predicción")
plt.ylabel("Etiqueta Real")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_PATH, "confusion_matrix.png"))
plt.close()

from sklearn.metrics import confusion_matrix
import seaborn as sns

cm = confusion_matrix(y_test, final_test_preds)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.title("Matriz de Confusión - Modelo Promediado")
plt.xlabel("Predicción")
plt.ylabel("Etiqueta Real")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_PATH, "confusion_matrix.png"))
plt.close()

plt.figure()
plt.bar(local_df["subset"], local_df["avg_acc"])
plt.xlabel("Subconjunto")
plt.ylabel("Accuracy Promedio Local")
plt.title("Rendimiento promedio por nodo")
plt.grid(True)
plt.savefig(os.path.join(SAVE_PATH, "accuracy_per_node.png"))
plt.close()

plt.figure()
plt.bar(local_df["subset"], local_df["avg_loss"])
plt.xlabel("Subconjunto")
plt.ylabel("Loss Promedio")
plt.title("Pérdida promedio por nodo")
plt.grid(True)
plt.savefig(os.path.join(SAVE_PATH, "loss_per_node.png"))
plt.close()

plt.figure()
plt.bar(local_df["subset"], local_df["avg_grad_norm"])
plt.xlabel("Subconjunto")
plt.ylabel("||Grad|| Promedio")
plt.title("Norma promedio del gradiente por nodo")
plt.grid(True)
plt.savefig(os.path.join(SAVE_PATH, "grad_norm_per_node.png"))
plt.close()


# Dispersión de parámetros antes del promedio
distances = []

for m in local_models:
    w1_k, b1_k, w2_k, b2_k = m

    dist = (
        np.linalg.norm(w1_k - w1_avg) +
        np.linalg.norm(w2_k - w2_avg) +
        np.linalg.norm(b1_k - b1_avg) +
        np.linalg.norm(b2_k - b2_avg)
    )

    distances.append(dist)

plt.figure()
plt.bar(range(1, N_SUBSETS+1), distances)
plt.xlabel("Subconjunto")
plt.ylabel("Distancia al modelo promedio")
plt.title("Dispersión de parámetros antes del promedio")
plt.grid(True)
plt.savefig(os.path.join(SAVE_PATH, "parameter_dispersion.png"))
plt.close()

# Accuracy del modelo final vs promedio local
plt.figure()
plt.axhline(test_acc, linestyle='--', label='Modelo Promediado')
plt.bar(local_df["subset"], local_df["avg_acc"], alpha=0.6)
plt.xlabel("Subconjunto")
plt.ylabel("Accuracy")
plt.title("Comparación: Nodos vs Modelo Final")
plt.legend()
plt.grid(True)
plt.savefig(os.path.join(SAVE_PATH, "node_vs_final_accuracy.png"))
plt.close()

sample_indices = np.random.choice(len(X_test), 10, replace=False)
predictions = predict(X_test[sample_indices])

for i, idx in enumerate(sample_indices):
    plt.subplot(2, 10, i+1)
    plt.imshow(X_test[idx].reshape(28, 28), cmap='gray')
    plt.title(f"P:{predictions[i]} R:{y_test[idx]}")
    plt.axis('off')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_PATH, "sample_predictions.png"))
plt.close()



INICIANDO ENTRENAMIENTO - MODEL AVERAGING
Run: 2
Seed: 44
Subconjuntos (nodos): 10
Muestras totales: 60000

--- Entrenando Subconjunto 1/10 ---
Subset 1 | Update 1 | Loss 0.0900 | Acc 0.1112
Subset 1 | Update 50 | Loss 0.0899 | Acc 0.1838
Subset 1 | Update 100 | Loss 0.0895 | Acc 0.1732
Subset 1 | Update 150 | Loss 0.0865 | Acc 0.2628
Subset 1 | Update 200 | Loss 0.0754 | Acc 0.4220
Subset 1 | Update 250 | Loss 0.0609 | Acc 0.5855
Subset 1 | Update 300 | Loss 0.0483 | Acc 0.7173
Subset 1 | Update 350 | Loss 0.0395 | Acc 0.7637
Subset 1 | Update 400 | Loss 0.0340 | Acc 0.7930

--- Entrenando Subconjunto 2/10 ---
Subset 2 | Update 1 | Loss 0.0900 | Acc 0.1140
Subset 2 | Update 50 | Loss 0.0899 | Acc 0.1185
Subset 2 | Update 100 | Loss 0.0893 | Acc 0.2982
Subset 2 | Update 150 | Loss 0.0854 | Acc 0.3007
Subset 2 | Update 200 | Loss 0.0741 | Acc 0.4503
Subset 2 | Update 250 | Loss 0.0583 | Acc 0.6125
Subset 2 | Update 300 | Loss 0.0457 | Acc 0.7277
Subset 2 | Update 350 | Loss 0.0374 | Ac

# Método 2

In [16]:
print("\n" + "="*50)
print("INICIANDO ENTRENAMIENTO")
print(f"Metodo: Método 2")
print(f"Run: {RUN_ID}")
print(f"Seed: {SEED}")
print("="*50)
print(f"Épocas: {EPOCHS}")
print(f"Learning Rate: {LEARNING_RATE}")
print(f"Muestras de entrenamiento: {len(X_train)}")
print(f"Entrenando con Batch Size: {BATCH_SIZE} (Total batches por época: {N_BATCHES})")
print("="*50)

SAVE_PATH = get_save_path("Metodo_2")

# Historial para graficar
train_losses = []
train_accuracies = []
test_accuracies = []
grad_norms = []

w1, b1 = w1_init.copy(), b1_init.copy()
w2, b2 = w2_init.copy(), b2_init.copy()

for epoch in range(EPOCHS):
    epoch_loss = 0
    epoch_accuracy = 0
    epoch_grad_norm = 0

    # Barajar datos al inicio de cada época
    indices = np.random.permutation(len(X_train))
    X_shuffled = X_train[indices]
    y_shuffled = y_train[indices]

    # Entrenar por batches
    for i in range(N_BATCHES):
        start_idx = i * BATCH_SIZE
        end_idx = start_idx + BATCH_SIZE

        X_batch = X_shuffled[start_idx:end_idx]
        y_batch = y_shuffled[start_idx:end_idx]

        loss, acc, grad_norm = train_batch(X_batch, y_batch, w1, w2, b1, b2)

        epoch_loss += loss
        epoch_accuracy += acc
        epoch_grad_norm += grad_norm


    # Promedios de la época
    avg_loss = epoch_loss / N_BATCHES
    avg_train_acc = epoch_accuracy / N_BATCHES
    avg_grad_norm = epoch_grad_norm / N_BATCHES

    # Evaluar en test set (sin entrenar, solo forward)
    test_preds = predict(X_test)
    test_acc = np.mean(test_preds == y_test)

    train_losses.append(avg_loss)
    train_accuracies.append(avg_train_acc)
    test_accuracies.append(test_acc)
    grad_norms.append(avg_grad_norm)

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Época {epoch+1:2d}/{EPOCHS} | "
            f"Pérdida: {avg_loss:.4f} | "
            f"Train Acc: {avg_train_acc:.4f} | "
            f"Test Acc: {test_acc:.4f}")


config_df = pd.DataFrame({
    "RUN_ID": [RUN_ID],
    "EPOCHS": [EPOCHS],
    "LEARNING_RATE": [LEARNING_RATE],
    "BATCH_SIZE": [BATCH_SIZE],
    "TOTAL_UPDATES": [EPOCHS * N_BATCHES],
    "SEED": [SEED]
})

config_df.to_csv(os.path.join(SAVE_PATH, "config.csv"), index=False)

metrics_df = pd.DataFrame({
    "epoch": range(1, EPOCHS + 1),
    "train_loss": train_losses,
    "train_accuracy": train_accuracies,
    "test_accuracy": test_accuracies,
    "grad_norm": grad_norms
})

metrics_df.to_excel(os.path.join(SAVE_PATH, "metrics.xlsx"), index=False)

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.plot(train_losses, linewidth=2)
plt.title('Pérdida')
plt.xlabel('Época')
plt.ylabel('MSE')
plt.grid(True)

plt.subplot(1, 3, 2)
plt.plot(train_accuracies, label='Train', linewidth=2)
plt.plot(test_accuracies, label='Test', linewidth=2)
plt.legend()
plt.title('Accuracy')
plt.xlabel('Época')
plt.grid(True)

plt.subplot(1, 3, 3)
plt.plot(grad_norms, linewidth=2)
plt.title('Norma del gradiente')
plt.xlabel('Época')
plt.ylabel('||grad||')
plt.grid(True)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_PATH, "training_curves.png"))
plt.close()

plt.figure(figsize=(15, 3))

sample_indices = np.random.choice(len(X_test), 10, replace=False)
predictions = predict(X_test[sample_indices])

for i, idx in enumerate(sample_indices):
    plt.subplot(2, 10, i+1)
    plt.imshow(X_test[idx].reshape(28, 28), cmap='gray')
    plt.title(f"P:{predictions[i]} R:{y_test[idx]}")
    plt.axis('off')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_PATH, "sample_predictions.png"))
plt.close()

# Precisión final
final_train_preds = predict(X_train)
final_test_preds = predict(X_test)

train_acc = np.mean(final_train_preds == y_train)
test_acc = np.mean(final_test_preds == y_test)

evaluation_df = pd.DataFrame({
    "train_accuracy_final": [train_acc],
    "test_accuracy_final": [test_acc]
})

evaluation_df.to_csv(os.path.join(SAVE_PATH, "evaluation.csv"), index=False)

# -------------------------------
# Matriz de confusión
# -------------------------------
cm = confusion_matrix(y_test, final_test_preds)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.title("Matriz de Confusión - Test Set")
plt.xlabel("Predicción")
plt.ylabel("Etiqueta Real")

plt.tight_layout()
plt.savefig(os.path.join(SAVE_PATH, "confusion_matrix.png"))
plt.close()


# -------------------------------
# Top 10 errores más comunes
# -------------------------------
from collections import Counter
errors = Counter()
for i in range(len(y_test)):
    if final_test_preds[i] != y_test[i]:
        errors[f"{y_test[i]}→{final_test_preds[i]}"] += 1

error_list = []
for error, count in errors.most_common():
    porcentaje = count / len(y_test) * 100
    error_list.append({
        "error": error,
        "count": count,
        "percentage": porcentaje
    })

errors_df = pd.DataFrame(error_list)
errors_df.to_csv(os.path.join(SAVE_PATH, "top_errors.csv"), index=False)



INICIANDO ENTRENAMIENTO
Metodo: Método 2
Run: 2
Seed: 44
Épocas: 200
Learning Rate: 0.05
Muestras de entrenamiento: 60000
Entrenando con Batch Size: 6000 (Total batches por época: 10)
Época  1/200 | Pérdida: 0.0900 | Train Acc: 0.1447 | Test Acc: 0.1220
Época 10/200 | Pérdida: 0.0895 | Train Acc: 0.2978 | Test Acc: 0.2997
Época 20/200 | Pérdida: 0.0762 | Train Acc: 0.4222 | Test Acc: 0.4376
Época 30/200 | Pérdida: 0.0488 | Train Acc: 0.7131 | Test Acc: 0.7269
Época 40/200 | Pérdida: 0.0338 | Train Acc: 0.7972 | Test Acc: 0.8060
Época 50/200 | Pérdida: 0.0276 | Train Acc: 0.8296 | Test Acc: 0.8341
Época 60/200 | Pérdida: 0.0242 | Train Acc: 0.8507 | Test Acc: 0.8526
Época 70/200 | Pérdida: 0.0218 | Train Acc: 0.8642 | Test Acc: 0.8656
Época 80/200 | Pérdida: 0.0202 | Train Acc: 0.8745 | Test Acc: 0.8764
Época 90/200 | Pérdida: 0.0190 | Train Acc: 0.8815 | Test Acc: 0.8838
Época 100/200 | Pérdida: 0.0180 | Train Acc: 0.8874 | Test Acc: 0.8887
Época 110/200 | Pérdida: 0.0172 | Train Acc:

# Método prueba

In [17]:
print("\n" + "="*50)
print("INICIANDO ENTRENAMIENTO")
print(f"Método: Método 1")
print(f"Run: {RUN_ID}")
print(f"Seed: {SEED}")
print("="*50)
print(f"Épocas: {EPOCHS}")
print(f"Learning Rate: {LEARNING_RATE}")
print(f"Muestras de entrenamiento: {len(X_train)}")
print(f"Entrenando con Batch Size: {BATCH_SIZE} (Total batches por época: {N_BATCHES})")
print("="*50)

SAVE_PATH = get_save_path("Metodo_1_prueba_de_concepto")

# Historial para graficar
train_losses = []
train_accuracies = []
test_accuracies = []
grad_norms = []

w1, b1 = w1_init.copy(), b1_init.copy()
w2, b2 = w2_init.copy(), b2_init.copy()

MAX_UPDATES_PER_BATCH = 100
TARGET_ACC = 0.8

for epoch in range(EPOCHS):

    epoch_loss = 0
    epoch_accuracy = 0
    epoch_grad_norm = 0
    total_updates = 0

    indices = np.random.permutation(len(X_train))
    X_shuffled = X_train[indices]
    y_shuffled = y_train[indices]

    for i in range(N_BATCHES):

        start_idx = i * BATCH_SIZE
        end_idx = start_idx + BATCH_SIZE

        X_batch = X_shuffled[start_idx:end_idx]
        y_batch = y_shuffled[start_idx:end_idx]

        batch_acc = 0
        j = 0

        while batch_acc < TARGET_ACC and j < MAX_UPDATES_PER_BATCH:

            loss, acc, grad_norm = train_batch(
                X_batch, y_batch, w1, w2, b1, b2
            )

            batch_acc = acc
            j += 1
            total_updates += 1

            epoch_loss += loss
            epoch_accuracy += acc
            epoch_grad_norm += grad_norm

    avg_loss = epoch_loss / total_updates
    avg_train_acc = epoch_accuracy / total_updates
    avg_grad_norm = epoch_grad_norm / total_updates


    # Evaluar en test set (sin entrenar, solo forward)
    test_preds = predict(X_test)
    test_acc = np.mean(test_preds == y_test)

    train_losses.append(avg_loss)
    train_accuracies.append(avg_train_acc)
    test_accuracies.append(test_acc)
    grad_norms.append(avg_grad_norm)

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Época {epoch+1:2d}/{EPOCHS} | "
            f"Pérdida: {avg_loss:.4f} | "
            f"Train Acc: {avg_train_acc:.4f} | "
            f"Test Acc: {test_acc:.4f}")


config_df = pd.DataFrame({
    "RUN_ID": [RUN_ID],
    "EPOCHS": [EPOCHS],
    "LEARNING_RATE": [LEARNING_RATE],
    "BATCH_SIZE": [BATCH_SIZE],
    "TOTAL_UPDATES": [EPOCHS * N_BATCHES],
    "SEED": [SEED]
})

config_df.to_csv(os.path.join(SAVE_PATH, "config.csv"), index=False)

metrics_df = pd.DataFrame({
    "epoch": range(1, EPOCHS + 1),
    "train_loss": train_losses,
    "train_accuracy": train_accuracies,
    "test_accuracy": test_accuracies,
    "grad_norm": grad_norms
})

metrics_df.to_excel(os.path.join(SAVE_PATH, "metrics.xlsx"), index=False)

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.plot(train_losses, linewidth=2)
plt.title('Pérdida')
plt.xlabel('Época')
plt.ylabel('MSE')
plt.grid(True)

plt.subplot(1, 3, 2)
plt.plot(train_accuracies, label='Train', linewidth=2)
plt.plot(test_accuracies, label='Test', linewidth=2)
plt.legend()
plt.title('Accuracy')
plt.xlabel('Época')
plt.grid(True)

plt.subplot(1, 3, 3)
plt.plot(grad_norms, linewidth=2)
plt.title('Norma del gradiente')
plt.xlabel('Época')
plt.ylabel('||grad||')
plt.grid(True)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_PATH, "training_curves.png"))
plt.close()

plt.figure(figsize=(15, 3))

sample_indices = np.random.choice(len(X_test), 10, replace=False)
predictions = predict(X_test[sample_indices])

for i, idx in enumerate(sample_indices):
    plt.subplot(2, 10, i+1)
    plt.imshow(X_test[idx].reshape(28, 28), cmap='gray')
    plt.title(f"P:{predictions[i]} R:{y_test[idx]}")
    plt.axis('off')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_PATH, "sample_predictions.png"))
plt.close()

# Precisión final
final_train_preds = predict(X_train)
final_test_preds = predict(X_test)

train_acc = np.mean(final_train_preds == y_train)
test_acc = np.mean(final_test_preds == y_test)

evaluation_df = pd.DataFrame({
    "train_accuracy_final": [train_acc],
    "test_accuracy_final": [test_acc]
})

evaluation_df.to_csv(os.path.join(SAVE_PATH, "evaluation.csv"), index=False)

# -------------------------------
# Matriz de confusión
# -------------------------------
cm = confusion_matrix(y_test, final_test_preds)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.title("Matriz de Confusión - Test Set")
plt.xlabel("Predicción")
plt.ylabel("Etiqueta Real")

plt.tight_layout()
plt.savefig(os.path.join(SAVE_PATH, "confusion_matrix.png"))
plt.close()


# -------------------------------
# Top 10 errores más comunes
# -------------------------------
from collections import Counter
errors = Counter()
for i in range(len(y_test)):
    if final_test_preds[i] != y_test[i]:
        errors[f"{y_test[i]}→{final_test_preds[i]}"] += 1

error_list = []
for error, count in errors.most_common():
    porcentaje = count / len(y_test) * 100
    error_list.append({
        "error": error,
        "count": count,
        "percentage": porcentaje
    })

errors_df = pd.DataFrame(error_list)
errors_df.to_csv(os.path.join(SAVE_PATH, "top_errors.csv"), index=False)



INICIANDO ENTRENAMIENTO
Método: Método 1
Run: 2
Seed: 44
Épocas: 200
Learning Rate: 0.05
Muestras de entrenamiento: 60000
Entrenando con Batch Size: 6000 (Total batches por época: 10)
Época  1/200 | Pérdida: 0.0665 | Train Acc: 0.4922 | Test Acc: 0.8136
Época 10/200 | Pérdida: 0.0266 | Train Acc: 0.8352 | Test Acc: 0.8394
Época 20/200 | Pérdida: 0.0236 | Train Acc: 0.8532 | Test Acc: 0.8552
Época 30/200 | Pérdida: 0.0215 | Train Acc: 0.8657 | Test Acc: 0.8674
Época 40/200 | Pérdida: 0.0200 | Train Acc: 0.8756 | Test Acc: 0.8771
Época 50/200 | Pérdida: 0.0188 | Train Acc: 0.8823 | Test Acc: 0.8829
Época 60/200 | Pérdida: 0.0179 | Train Acc: 0.8879 | Test Acc: 0.8886
Época 70/200 | Pérdida: 0.0172 | Train Acc: 0.8919 | Test Acc: 0.8916
Época 80/200 | Pérdida: 0.0165 | Train Acc: 0.8953 | Test Acc: 0.8943
Época 90/200 | Pérdida: 0.0160 | Train Acc: 0.8983 | Test Acc: 0.8986
Época 100/200 | Pérdida: 0.0156 | Train Acc: 0.9004 | Test Acc: 0.9021
Época 110/200 | Pérdida: 0.0151 | Train Acc: